## Imports

In [1]:
# | code-fold: true
# | code-summary: "Load packages"
# | output: false


import os
import numpy as np
import os
import numpy as np
from sympy import Matrix, sqrt, Piecewise
import sympy as sp
import pytest
from attr import define, field
from sympy import MutableDenseNDimArray as Arr


from zoomy_core.fvm.solver_numpy import Settings
from zoomy_core.model.basemodel import Model, eigenvalue_dict_to_matrix
import zoomy_core.model.initial_conditions as IC
import zoomy_core.model.boundary_conditions as BC
from zoomy_core.misc.misc import Zstruct, ZArray
import zoomy_core.misc.misc as misc
import zoomy_firedrake.firedrake_solver as dg


In [ ]:
@define(frozen=True, slots=True, kw_only=True)
class SWE(Model):
    dimension: int = 2
    variables: Zstruct = field(init=False)
    aux_variables: Zstruct = field(default=1)
    _default_parameters: dict = field(
        init=False, factory=lambda: {"g": 9.81, "ex": 0.0, "ey": 0.0, "ez": 1.0, "rho": 1000.0, "n": 0.1, "eps":1e-4}
    )
    
    def __attrs_post_init__(self):
        object.__setattr__(self, "variables", self.dimension + 2)
        super().__attrs_post_init__()

    def project_2d_to_3d(self):
        out = ZArray.zeros(6)
        p = self.parameters
        dim = self.dimension
        z = self.position[2]
        b = self.aux_variables[0]
        h = self.variables[1]
        U = [hu / h for hu in self.variables[2 : 2 + dim]]
        out[0] = b
        out[1] = h
        out[2] = U[0]
        out[3] = 0 if dim == 1 else U[1]
        out[4] = 0
        out[5] = p.rho * p.g * h * (1 - z)
        return out
    
    def get_primitives(self):
        dim = self.dimension
        b = self.variables[0]
        h = self.variables[1]
        hinv = 1/h
        U = Matrix([hu * hinv for hu in self.variables[2 : 2 + dim]])
        return b, h, U, hinv

    def flux(self):
        dim = self.dimension
        b, h, U, hinv = self.get_primitives()
        g = self.parameters.g
        I = Matrix.eye(dim)
        F = Matrix.zeros(self.variables.length(), dim)
        F[1, :] = sp.Matrix(self.variables[2: 2 + dim]).T
        F[2:, :] = h * U * U.T
        return ZArray(F)
    
    def nonconservative_matrix(self):
        dim = self.dimension
        b, h, U, hinv = self.get_primitives()
        U = Matrix([hu * hinv for hu in self.variables[2 : 2 + dim]])
        g = self.parameters.g
        N = ZArray.zeros(self.n_variables, self.n_variables, dim)
        for d in range(dim):
            N[2+d, 0, d] = g * h # g * h * grad(b)
            N[2+d, 1, d] = g * h # g * h * grad(h)
        return ZArray(N)
    
    def source(self):
        eps = 1e-4
        dim = self.dimension
        _, _, U, _ = self.get_primitives()
        hinv = self.aux_variables[0]
        g = self.parameters.g
        n = self.parameters.n
        abs_u = sqrt(U.dot(U) + eps)
        S = Matrix.zeros(self.n_variables, 1)
        S[2:, 0] = n**2 * g  * (hinv**(1/3) + eps) * U[:, 0] * abs_u
        return ZArray(S).reshape(self.n_variables,)
    
@define(frozen=True, slots=True, kw_only=True)
class NumericSWE(SWE):
    disable_differentiation: bool = False
    
    def get_primitives(self):
        dim = self.dimension
        b = self.variables[0]
        h = self.variables[1]
        hinv = self.aux_variables[0]
        U = Matrix([hu * hinv for hu in self.variables[2 : 2 + dim]])
        
        return b, h, U, hinv
    
    def eigenvalues(self):
        ev = super().eigenvalues()
        h = self.variables[1]
        return sp.Function('conditional')(h > self.parameters.eps, ev, ZArray.zeros(*ev.shape))
    
    
    def source(self):
        delta = self.parameters.eps  # or smaller
        h = self.variables[1]
        smooth = sp.Rational(1,2)*(1 + sp.tanh((h - self.parameters.eps)/delta))

        S = super().source()
        S2 = sp.Matrix(S)
        S2 = S2.subs({h: self.parameters.eps})
        Sreg = ZArray.zeros(*S.shape)
        for i in range(self.n_variables):
            Sreg[i] = S2[i,0]
        zeros = ZArray.zeros(*S.shape)
        return sp.Function('conditional')(h > self.parameters.eps, -S, zeros)
    
    def source_jacobian_wrt_aux_variables(self):
        return ZArray.zeros(
            self.n_variables
        )
    
    def source_jacobian_wrt_variables(self):
        return ZArray.zeros(
            self.n_variables
        )
                



# Transformation to UFL Code (Medium)

### Map from Sympy to UFL

In [ ]:


bcs = BC.BoundaryConditions(
    [
        BC.Extrapolation(tag="wall"),
        BC.Extrapolation(tag="inflow"),
        BC.Extrapolation(tag="outflow"),
    ]
)

 ### Initial condition
def ic_q(x):
    R = 3
    r = np.sqrt((x[0])**2 + (x[1])**2)
    b = r**2 / 100 * 3
    h = np.where(r <= R, 2., 1) -b
    h = np.where(h <= 0, 0, h)
    return np.array([b, h , 0.*x[0], 0.*x[0]])

ic = IC.UserFunction(ic_q)

model = NumericSWE(
    dimension=2,
    boundary_conditions=bcs,
    initial_conditions=ic,
)

settings = Settings(name="Firedrake", output=Zstruct(directory="outputs/firedrake", snapshots=1000, filename='dg', clean_directory=True))


In [4]:
import ufl 
IdentityMatrix = ufl.as_tensor([[0, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]])

solver = dg.FiredrakeHyperbolicSolver(settings=settings, time_end = 10.0, CFL=0.45, IdentityMatrix=IdentityMatrix)

In [5]:
main_dir = misc.get_main_directory()
path_to_mesh = os.path.join(main_dir, "meshes", "square", "mesh.msh")
solver.solve(path_to_mesh, model)

2025-11-16 11:16:15.816 | INFO     | zoomy_firedrake.firedrake_solver:solve:515 - iteration: 10, time: 0.179755, dt: 0.017987, next write at time: 0.110110
2025-11-16 11:16:16.983 | INFO     | zoomy_firedrake.firedrake_solver:solve:515 - iteration: 20, time: 0.359815, dt: 0.018035, next write at time: 0.210210
2025-11-16 11:16:18.183 | INFO     | zoomy_firedrake.firedrake_solver:solve:515 - iteration: 30, time: 0.540717, dt: 0.018111, next write at time: 0.310310
2025-11-16 11:16:19.306 | INFO     | zoomy_firedrake.firedrake_solver:solve:515 - iteration: 40, time: 0.722767, dt: 0.018330, next write at time: 0.410410
2025-11-16 11:16:20.557 | INFO     | zoomy_firedrake.firedrake_solver:solve:515 - iteration: 50, time: 0.908890, dt: 0.018892, next write at time: 0.510511
2025-11-16 11:16:22.020 | INFO     | zoomy_firedrake.firedrake_solver:solve:515 - iteration: 60, time: 1.102362, dt: 0.019764, next write at time: 0.610611
2025-11-16 11:16:23.440 | INFO     | zoomy_firedrake.firedrake_s